# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
# YOUR CODE HERE
from google.colab import drive
drive.mount('/content/drive')

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

import json
import glob
import os

DRIVE_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts"

# ------------------------------------------------------------
# Load outputs.json from Notebook 3
# ------------------------------------------------------------
with open(DRIVE_DIR + "/outputs.json", "r") as f:
    results = json.load(f)

print("Loaded outputs.json:", len(results), "records")
print("Sample record keys:", list(results[0].keys()))
print("")
print(results[0])

# ------------------------------------------------------------
# Verify corporate_policies folder is accessible
# ------------------------------------------------------------
POLICY_DIR = "corporate_policies"

if not os.path.isdir(POLICY_DIR):
    print("")
    print("corporate_policies not found locally - check if it needs to be")
    print("re-uploaded/re-copied for this session, or if it lives in Drive too.")
else:
    policy_files = sorted(glob.glob(os.path.join(POLICY_DIR, "*.md")))
    print("")
    print("Found", len(policy_files), "policy files:")
    for f in policy_files:
        print("  -", os.path.basename(f))

# ============================================================
# Task: Initialise all-MiniLM-L6-v2 embedding model, embed the
# SOP documents, and validate embeddings were produced
# ============================================================

!pip install -q langchain langchain-community sentence-transformers

from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"  # must match NB5, NB7

# ------------------------------------------------------------
# 1. Load SOP documents (reusing the TextLoader approach from NB2)
# ------------------------------------------------------------
documents = []

for fpath in policy_files:
    loader = TextLoader(fpath, encoding="utf-8")
    loaded_docs = loader.load()

    for doc in loaded_docs:
        doc.metadata["source_file"] = os.path.basename(fpath)
        doc.metadata["policy_name"] = os.path.splitext(os.path.basename(fpath))[0]

    documents.extend(loaded_docs)

print("")
print("Loaded", len(documents), "policy documents for embedding")

# ------------------------------------------------------------
# 2. Initialise the embedding model (runs on CPU, no GPU needed)
# ------------------------------------------------------------
print("")
print("Loading embedding model:", EMBEDDING_MODEL_ID)

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={"device": "cpu"}
)

print("Embedding model loaded.")

# ------------------------------------------------------------
# 3. Validate: embed one sample string, check vector length
# ------------------------------------------------------------
sample_text = "How long does shipping take for domestic orders?"
sample_vector = embedding_model.embed_query(sample_text)

print("")
print("--- Embedding validation ---")
print("Sample text:", sample_text)
print("Vector length:", len(sample_vector))
print("Expected dimension for MiniLM: 384")
print("Match:", len(sample_vector) == 384)

assert len(sample_vector) == 384, "Embedding dimension mismatch - expected 384 for all-MiniLM-L6-v2"

# ------------------------------------------------------------
# 4. Embed all SOP documents
# ------------------------------------------------------------
policy_texts = [doc.page_content for doc in documents]
policy_embeddings = embedding_model.embed_documents(policy_texts)

print("")
print("--- SOP document embeddings ---")
print("Number of documents embedded:", len(policy_embeddings))
print("Embedding dimension per document:", len(policy_embeddings[0]))

# Confirm every document produced a valid, correctly-sized vector
all_correct_dim = all(len(vec) == 384 for vec in policy_embeddings)
print("All embeddings have correct dimension (384):", all_correct_dim)

any_zero_vectors = any(all(v == 0 for v in vec) for vec in policy_embeddings)
print("Any all-zero (failed) embeddings:", any_zero_vectors)

assert all_correct_dim, "Some embeddings have incorrect dimension!"
assert not any_zero_vectors, "Some embeddings are all-zero - embedding may have failed silently!"

print("")
print("Embedding validation PASSED -", len(documents), "documents embedded at 384 dimensions each.")

# ------------------------------------------------------------
# 5. Per-document summary
# ------------------------------------------------------------
print("")
print("--- Per-document embedding summary ---")
for doc, vec in zip(documents, policy_embeddings):
    print(doc.metadata["policy_name"], "- vector dim:", len(vec), "- first 3 values:", [round(v, 4) for v in vec[:3]])

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
Loaded outputs.json: 391 records
Sample record keys: ['test_query', 'ground_truth', 'baseline_output']

{'test_query': 'need help to shop several of ur item', 'ground_truth': '{"intent":"place_order","category":"ORDER"}', 'baseline_output': '```json\n{\n  "intent": "customer_support",\n  "category": "shopping"\n}\n```'}

Found 13 policy files:
  - account_recovery.md
  - billing_disputes.md
  - data_privacy.md
  - escalation_matrix.md
  - order_tracking.md
  - password_reset.md
  - payment_methods.md
  - product_return.md
  - refund_policy.md
  - shipping_delays.md
  - subscription_cancellation.md
  - technical_troubleshooting.md
  - working_hours.md
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1

/tmp/ipykernel_4849/420720124.py:49: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader



Loaded 13 policy documents for embedding

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


/tmp/ipykernel_4849/420720124.py:78: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.

--- Embedding validation ---
Sample text: How long does shipping take for domestic orders?
Vector length: 384
Expected dimension for MiniLM: 384
Match: True

--- SOP document embeddings ---
Number of documents embedded: 13
Embedding dimension per document: 384
All embeddings have correct dimension (384): True
Any all-zero (failed) embeddings: False

Embedding validation PASSED - 13 documents embedded at 384 dimensions each.

--- Per-document embedding summary ---
account_recovery - vector dim: 384 - first 3 values: [-0.0408, 0.0618, 0.0398]
billing_disputes - vector dim: 384 - first 3 values: [-0.1094, 0.0325, 0.046]
data_privacy - vector dim: 384 - first 3 values: [-0.1209, 0.0629, -0.0274]
escalation_matrix - vector dim: 384 - first 3 values: [-0.0418, -0.0143, -0.0251]
order_tracking - vector dim: 384 - first 3 values: [-0.029, -0.0279, 0.0043]
password_reset - vector dim: 384 - first 3 values: [-0.0944, -0.0517, -0.0023]
payment_methods - vector dim: 384 - 

#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [2]:
# YOUR CODE HERE
!pip install -q chromadb langchain-chroma

from langchain_chroma import Chroma

PERSIST_DIR = "./chroma_db"

# ------------------------------------------------------------
# 1. Create the persistent Chroma index from the embedded documents
#    (from_documents re-embeds internally using the embedding_model
#    object, using the same 'documents' list built in the previous step)
# ------------------------------------------------------------
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR
)

print("Chroma index created and persisted to:", PERSIST_DIR)

# ------------------------------------------------------------
# 2. Validate: collection count should equal number of SOP documents
# ------------------------------------------------------------
collection_count = vector_db._collection.count()

print("")
print("--- Index validation ---")
print("Documents embedded:", len(documents))
print("Vectors in Chroma collection:", collection_count)
print("Match:", collection_count == len(documents))

assert collection_count == len(documents), "Mismatch between document count and Chroma collection count!"

# ------------------------------------------------------------
# 3. Test similarity search
# ------------------------------------------------------------
test_query = "refund"
search_results = vector_db.similarity_search(test_query, k=1)

print("")
print("--- Test similarity search: query =", "'" + test_query + "'", "---")
print("Top result source:", search_results[0].metadata["policy_name"])
print("Content preview:", search_results[0].page_content[:200])

# ------------------------------------------------------------
# 4. Additional spot-checks across a few different topics,
#    to confirm retrieval is topically accurate, not just functional
# ------------------------------------------------------------
spot_check_queries = [
    "how do I reset my password",
    "my order hasn't arrived",
    "cancel my subscription",
    "how do I dispute a charge",
]

print("")
print("--- Spot-check retrieval across topics ---")
for q in spot_check_queries:
    result = vector_db.similarity_search(q, k=1)
    print("")
    print("Query:", q)
    print("  Retrieved:", result[0].metadata["policy_name"])

# ------------------------------------------------------------
# 5. Confirm the persisted directory actually contains files on disk
# ------------------------------------------------------------
import os
persisted_files = os.listdir(PERSIST_DIR)
print("")
print("Files in", PERSIST_DIR, ":", persisted_files)
print("Persistence confirmed:", len(persisted_files) > 0)

print("")
print("ChromaDB index build and validation PASSED.")
print(collection_count, "documents indexed, similarity search functioning correctly.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [3]:
# YOUR CODE HERE
# ============================================================
# Load Qwen model for generation (embedding model already on CPU;
# this notebook now needs both loaded simultaneously)
# ============================================================

!pip install -q -U bitsandbytes accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # must match NB2, NB3 exactly

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

print("")
print("Model loaded. GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))


def generate_response(prompt_text, model, tokenizer, max_new_tokens=150):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


# ============================================================
# Naive RAG: retrieve top-1 doc, augment prompt, generate
# ============================================================

RAG_SYSTEM_PROMPT_TEMPLATE = (
    "You are a helpful customer support assistant. "
    "Answer strictly using this SOP:\n\n{context}\n\n"
    "If the SOP does not contain the answer, say you don't have that information. "
    "Be concise."
)

def build_rag_prompt(query, context, tokenizer):
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT_TEMPLATE.format(context=context)},
        {"role": "user", "content": query},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


K = 1  # top-1 retrieval per the task; try k=3 if SOPs overlap in practice

naive_rag_results = []
retrieval_log = []  # tracks which policy was retrieved per query, for auditing wrong-retrieval cases

import time
start_time = time.time()

for idx in range(len(results)):
    query = results[idx]["test_query"]

    retrieved_docs = vector_db.similarity_search(query, k=K)
    retrieved_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    retrieved_policy_names = [doc.metadata["policy_name"] for doc in retrieved_docs]

    rag_prompt = build_rag_prompt(query, retrieved_context, tokenizer)
    rag_output = generate_response(rag_prompt, model, tokenizer)

    results[idx]["naive_rag_output"] = rag_output

    retrieval_log.append({
        "query": query,
        "retrieved_policy": retrieved_policy_names[0] if retrieved_policy_names else None
    })

    if (idx + 1) % 50 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(results) - (idx + 1)) / rate
        print("Processed", idx + 1, "/", len(results), "- elapsed:", round(elapsed, 1), "s - est. remaining:", round(remaining, 1), "s")

total_time = time.time() - start_time
print("")
print("Naive RAG generation complete.")
print("Total time:", round(total_time, 1), "seconds")

# ------------------------------------------------------------
# Spot-check a few samples
# ------------------------------------------------------------
print("")
print("--- Sample naive RAG outputs ---")
for r in results[:3]:
    print("Query:", r["test_query"])
    print("Retrieved-and-generated answer:", r["naive_rag_output"])
    print("")

# ------------------------------------------------------------
# Retrieval audit: flag which policy each query pulled, so you
# can manually spot-check for wrong-policy retrieval on ambiguous
# queries per the task's explicit callout
# ------------------------------------------------------------
import pandas as pd
retrieval_df = pd.DataFrame(retrieval_log)
print("")
print("--- Retrieved policy distribution across all queries ---")
print(retrieval_df["retrieved_policy"].value_counts())

# ------------------------------------------------------------
# Save updated outputs.json (adds naive_rag_output field)
# ------------------------------------------------------------
import json
DRIVE_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts"
outputs_path = DRIVE_DIR + "/outputs.json"

with open(outputs_path, "w") as f:
    json.dump(results, f, indent=2)

print("")
print("Saved updated outputs.json (with naive_rag_output) to:", outputs_path)

# Reload check
with open(outputs_path, "r") as f:
    reloaded = json.load(f)
assert "naive_rag_output" in reloaded[0], "naive_rag_output field missing after save!"
print("Integrity check PASSED - naive_rag_output present in saved file.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.1 MB/s eta 0:00:00
Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded. GPU memory allocated (GB): 1.15
Processed 50 / 391 - elapsed: 222.8 s - est. remaining: 1519.3 s
Processed 100 / 391 - elapsed: 392.2 s - est. remaining: 1141.4 s
Processed 150 / 391 - elapsed: 615.6 s - est. remaining: 989.0 s
Processed 200 / 391 - elapsed: 829.0 s - est. remaining: 791.7 s
Processed 250 / 391 - elapsed: 1037.0 s - est. remaining: 584.8 s
Processed 300 / 391 - elapsed: 1248.5 s - est. remaining: 378.7 s
Processed 350 / 391 - elapsed: 1475.2 s - est. remaining: 172.8 s

Naive RAG generation complete.
Total time: 1631.5 seconds

--- Sample naive RAG outputs ---
Query: need help to shop several of ur item
Retrieved-and-generated answer: I'm sorry, but I am unable to assist with that. Please let me know if there's anything else I can help with.

Query: i need assistance trying to update the shipping address
Retrieved-and-generated answer: I'm sorry, but I currently do not have the ability to assist with updating shipping addresses. Please try contacting you

---
## Save Artifacts for Downstream Notebooks

In [4]:
# YOUR CODE HERE
# ============================================================
# Save all NB4 artifacts for downstream notebooks (NB5, NB7)
# ============================================================

import os
import json

DRIVE_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts"
os.makedirs(DRIVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. ChromaDB index — persist to Drive (local ./chroma_db is
#    ephemeral and will be lost when this runtime disconnects)
# ------------------------------------------------------------
import shutil

local_chroma_path = "./chroma_db"
drive_chroma_path = DRIVE_DIR + "/chroma_db"

assert os.path.isdir(local_chroma_path), "Local ./chroma_db not found - build the index first"

if os.path.isdir(drive_chroma_path):
    shutil.rmtree(drive_chroma_path)  # clean overwrite, avoid stale mixed state

shutil.copytree(local_chroma_path, drive_chroma_path)

print("Copied ChromaDB index to Drive:", drive_chroma_path)
print("Files:", os.listdir(drive_chroma_path))

# ------------------------------------------------------------
# 2. outputs.json (updated with naive_rag_output) — already saved
#    during the generation loop, but re-confirm here as the
#    definitive checkpoint for this notebook
# ------------------------------------------------------------
assert "results" in dir(), "results not found - run the naive RAG generation loop first"
assert all("naive_rag_output" in r for r in results), "Some records missing naive_rag_output!"

outputs_path = DRIVE_DIR + "/outputs.json"
with open(outputs_path, "w") as f:
    json.dump(results, f, indent=2)

print("")
print("Saved outputs.json (with naive_rag_output) to:", outputs_path)
print("Total records:", len(results))

# ------------------------------------------------------------
# 3. Integrity check: reload both artifacts fresh from Drive
#    to confirm they'll actually work when NB5/NB7 load them
#    in a brand-new session
# ------------------------------------------------------------
with open(outputs_path, "r") as f:
    reloaded_outputs = json.load(f)

assert len(reloaded_outputs) == len(results), "outputs.json reload size mismatch!"
assert "naive_rag_output" in reloaded_outputs[0], "naive_rag_output missing after reload!"
assert "baseline_output" in reloaded_outputs[0], "baseline_output missing - NB3 data lost!"

print("")
print("--- Reload verification ---")
print("outputs.json:", len(reloaded_outputs), "records, schema:", list(reloaded_outputs[0].keys()))
print("chroma_db files on Drive:", len(os.listdir(drive_chroma_path)))

print("")
print("All NB4 artifacts saved and verified:")
print("  -", drive_chroma_path)
print("  -", outputs_path)
print("")
print("NB5 and NB7 can now load chroma_db via:")
print("  Chroma(persist_directory='" + drive_chroma_path + "', embedding_function=embedding_model)")
print("(using the SAME embedding model: sentence-transformers/all-MiniLM-L6-v2)")

Copied ChromaDB index to Drive: /content/drive/MyDrive/customer_support_llm_project/artifacts/chroma_db
Files: ['chroma.sqlite3', '69741db8-4008-4846-a351-427f85021270']

Saved outputs.json (with naive_rag_output) to: /content/drive/MyDrive/customer_support_llm_project/artifacts/outputs.json
Total records: 391

--- Reload verification ---
outputs.json: 391 records, schema: ['test_query', 'ground_truth', 'baseline_output', 'naive_rag_output']
chroma_db files on Drive: 2

All NB4 artifacts saved and verified:
  - /content/drive/MyDrive/customer_support_llm_project/artifacts/chroma_db
  - /content/drive/MyDrive/customer_support_llm_project/artifacts/outputs.json

NB5 and NB7 can now load chroma_db via:
  Chroma(persist_directory='/content/drive/MyDrive/customer_support_llm_project/artifacts/chroma_db', embedding_function=embedding_model)
(using the SAME embedding model: sentence-transformers/all-MiniLM-L6-v2)


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**